In [1]:
#run spark.stop before running this notebook. some of the configs won't take if an active spark sessions is grabbed or running. 
#spark.stop()

In [2]:
import psutil
import time
import threading
import datetime
import json
import xgboost as xgb

In [3]:
import pyspark
import os
import pandas as pd
from torchvision.io import read_image
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torchvision import datasets, models
from torchvision.transforms import ToTensor
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import pyarrow
from io import BytesIO
import base64



from pyspark.sql.functions import udf, pandas_udf
from pyspark.sql.types import FloatType

import math
from statistics import NormalDist

In [4]:
class CNN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.model = torch.nn.Sequential(
            #Input = 3 x 32 x 32, Output = 32 x 32 x 32 
            #new input 3x600x375 output=
            torch.nn.Conv2d(in_channels = 3, out_channels = 6, kernel_size = 3, padding = 'same'),
            torch.nn.ReLU(),
            #Input = 32 x 32 x 32, Output = 32 x 16 x 16 
            torch.nn.MaxPool2d(kernel_size=2),

            #Input = 32 x 16 x 16, Output = 64 x 16 x 16 
            torch.nn.Conv2d(in_channels = 6, out_channels = 12, kernel_size = 3, padding = 'same'),
            torch.nn.ReLU(),
            #Input = 64 x 16 x 16, Output = 64 x 8 x 8 
            torch.nn.MaxPool2d(kernel_size=2),

            #Input = 64 x 8 x 8, Output = 64 x 8 x 8 
            torch.nn.Conv2d(in_channels = 12, out_channels = 24, kernel_size = 3, padding = 'same'),
            torch.nn.ReLU(),
            #Input = 64 x 8 x 8, Output = 64 x 4 x 4 
            torch.nn.MaxPool2d(kernel_size=2),

            #Input = 64 x 8 x 8, Output = 64 x 8 x 8 
            torch.nn.Conv2d(in_channels = 24, out_channels = 48, kernel_size = 3, padding = 'same'),
            torch.nn.ReLU(),
            #Input = 64 x 8 x 8, Output = 64 x 4 x 4 
            torch.nn.MaxPool2d(kernel_size=2),

            #Input = 64 x 8 x 8, Output = 64 x 8 x 8 
            torch.nn.Conv2d(in_channels = 48, out_channels = 96, kernel_size = 3, padding = 'same'),
            torch.nn.ReLU(),
            #Input = 64 x 8 x 8, Output = 64 x 4 x 4 
            torch.nn.MaxPool2d(kernel_size=2),

            torch.nn.Flatten(),

            torch.nn.Linear(96*11*18, 128),
            torch.nn.ReLU(),
            torch.nn.BatchNorm1d(128),




            #torch.nn.Linear(128, 128),
            #torch.nn.ReLU(),
            #torch.nn.BatchNorm1d(128),



            torch.nn.Linear(128, 2),
            torch.nn.ReLU(),
            #torch.nn.BatchNorm1d(2),
            #torch.nn.Dropout(0.2),


            torch.nn.Softmax(dim=1)

        )

    def forward(self, x):

            

        return self.model(x)

In [5]:
#tuning knob 1. put a very between 1 and 10, depending on sys cores. this also reparitions the df for sql queries
threads = 4

In [6]:
spark = pyspark.sql.SparkSession.builder.master(f"local[{threads}]").appName("silver").config("spark.executor.memory", "24g").config("spark.driver.memory", "24g").getOrCreate()

print(spark)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/23 23:05:31 WARN Utils: Your hostname, starfuryz2040-VirtualBox, resolves to a loopback address: 127.0.1.1; using 192.168.0.137 instead (on interface enp0s3)
25/11/23 23:05:31 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/23 23:05:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [7]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = CNN().to(device)
model.load_state_dict(torch.load('../mymodel', map_location=torch.device(device)))
model.eval() # Set the model to evaluation mode

bc_model = spark.sparkContext.broadcast(model)

In [8]:
#second_model
res18_model = models.resnet18(pretrained=True).to(device)
res18_model.fc = nn.Linear(res18_model.fc.in_features, 2 )
res18_model.load_state_dict(torch.load('../resnet18_model.pth', map_location=torch.device(device)))
res18_model.eval() # Set the model to evaluation mode

bc_res18_model = spark.sparkContext.broadcast(res18_model)

/home/starfuryz2040/PycharmProjects/mdpbenchmark/.venv/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/starfuryz2040/PycharmProjects/mdpbenchmark/.venv/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
2.5%

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /home/starfuryz2040/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100.0%


In [9]:
print(device)

cpu


NORMAL UDF (doesnt work in spark sql)

In [10]:
def predict_tensor(base64image):
    img_data = base64.b64decode(base64image)
    image = Image.open(BytesIO(img_data))
    tensor = ToTensor()(image).float()
    
    with torch.no_grad():  # Disable gradient calculation for inference
        prediction = model(tensor.unsqueeze(0).to(device))
        _, y_pred = torch.max(prediction.data, 1)
        
    return float(y_pred)
    

    
    

predict_udf = udf(predict_tensor, FloatType())

In [11]:
spark.udf.register("predict_udf", predict_udf)

VECTORIZED UDF (WORKS IN SPARK SQL)

In [12]:
@pandas_udf(returnType=FloatType())
def v_predict_tensor(base64image):
    base64image_decode = base64image.apply(lambda x: base64.b64decode(x))
    image_pil = base64image_decode.apply(lambda x: Image.open(BytesIO(x)))
    tensor_series = image_pil.apply(lambda x: ToTensor()(x).float())
    input_tensor = tensor_series.tolist()
    
    with torch.no_grad():  # Disable gradient calculation for inference
        predictions = model(torch.stack(input_tensor).to(device))
        _, y_pred = torch.max(predictions.data, 1)
    #return y_pred
    return pd.Series(y_pred.squeeze().tolist())


In [13]:
spark.udf.register("v_predict_udf", v_predict_tensor)

In [14]:
#tuning knob 2. put a .limit(20) at the end of the data set loads to reduce memory footprint for dev stuff 
#all_images_labeled = spark.read.parquet("./silver/all_images_labeled.parquet")
labeled_training_set = spark.read.parquet("./silver/labeled_training_set_no_tensor.parquet")
labeled_training_set.cache()
labeled_training_set.count()

998

In [17]:
labeled_training_set.schema

StructType([StructField('name', StringType(), True), StructField('imageData', StringType(), True), StructField('prelabeled_fraud', IntegerType(), True), StructField('id', StringType(), True), StructField('person_name', StringType(), True), StructField('address', StringType(), True), StructField('birthday', StringType(), True), StructField('gender', StringType(), True), StructField('ethnicity', StringType(), True), StructField('class', StringType(), True), StructField('issue_date', StringType(), True), StructField('expire_date', StringType(), True), StructField('eye_color', StringType(), True), StructField('hair_color', StringType(), True), StructField('is_donor', StringType(), True), StructField('is_veteran', StringType(), True), StructField('license_number', StringType(), True), StructField('height_inches', IntegerType(), True), StructField('weight_lbs', StringType(), True)])

In [15]:
labeled_training_set.describe().show()

25/11/23 23:11:08 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+--------------------+--------------------+-------------------+--------------------+---------------+--------------------+----------+------+---------+-----+----------+-----------+---------+----------+--------+----------+--------------+-----------------+-----------------+
|summary|                name|           imageData|   prelabeled_fraud|                  id|    person_name|             address|  birthday|gender|ethnicity|class|issue_date|expire_date|eye_color|hair_color|is_donor|is_veteran|license_number|    height_inches|       weight_lbs|
+-------+--------------------+--------------------+-------------------+--------------------+---------------+--------------------+----------+------+---------+-----+----------+-----------+---------+----------+--------+----------+--------------+-----------------+-----------------+
|  count|                 998|                 998|                998|                 998|            998|                 998|       998|   998|      898|  998|

In [16]:
labeled_training_set.toPandas().describe()

,prelabeled_fraud,height_inches
count,998.000000,998.000000
mean,0.199399,72.203407
std,0.399749,10.146752
min,0.000000,0.000000
25%,0.000000,65.000000
50%,0.000000,70.000000
75%,0.000000,79.000000
max,1.000000,95.000000


In [18]:
labeled_training_set.toPandas().info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 998 entries, 0 to 997
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   name              998 non-null    object
 1   imageData         998 non-null    object
 2   prelabeled_fraud  998 non-null    int32 
 3   id                998 non-null    object
 4   person_name       998 non-null    object
 5   address           998 non-null    object
 6   birthday          998 non-null    object
 7   gender            998 non-null    object
 8   ethnicity         898 non-null    object
 9   class             998 non-null    object
 10  issue_date        998 non-null    object
 11  expire_date       998 non-null    object
 12  eye_color         998 non-null    object
 13  hair_color        898 non-null    object
 14  is_donor          998 non-null    object
 15  is_veteran        998 non-null    object
 16  license_number    998 non-null    object
 17  height_inches   

In [29]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

def get_string_length_stats(df):
    # 1. Identify all string columns
    string_cols = ['name',
 'imageData',
 'id',
 'person_name',
 'address',
 'birthday',
 'gender',
 'ethnicity',
 'class',
 'issue_date',
 'expire_date',
 'eye_color',
 'hair_color',
 'is_donor',
 'is_veteran',
 'license_number',
 'weight_lbs'
    ]

    if not string_cols:
        print("No string columns found in the DataFrame.")
        return None

    # 2. Create a list of aggregation expressions for each metric
    agg_exprs = []
    for col_name in string_cols:
        # Calculate length and then the statistics
        length_col = F.length(F.col(col_name))
        agg_exprs.append(F.avg(length_col).alias(f"{col_name}_avg_len"))
        agg_exprs.append(F.max(length_col).alias(f"{col_name}_max_len"))
        agg_exprs.append(F.min(length_col).alias(f"{col_name}_min_len"))
        # Use stddev for variance (stddev*stddev). PySpark has a var function alias too.
        agg_exprs.append(F.variance(length_col).alias(f"{col_name}_variance_len"))

    # 3. Apply all aggregations in a single .agg() call
    stats_df = df.agg(*agg_exprs)

    return stats_df

# Example Usage:
# Assuming you have a DataFrame named 'df'
# stats_result = get_string_length_stats(df)
# stats_result.show()


In [30]:
labeled_training_set.printSchema()

root
 |-- name: string (nullable = true)
 |-- imageData: string (nullable = true)
 |-- prelabeled_fraud: integer (nullable = true)
 |-- id: string (nullable = true)
 |-- person_name: string (nullable = true)
 |-- address: string (nullable = true)
 |-- birthday: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- ethnicity: string (nullable = true)
 |-- class: string (nullable = true)
 |-- issue_date: string (nullable = true)
 |-- expire_date: string (nullable = true)
 |-- eye_color: string (nullable = true)
 |-- hair_color: string (nullable = true)
 |-- is_donor: string (nullable = true)
 |-- is_veteran: string (nullable = true)
 |-- license_number: string (nullable = true)
 |-- height_inches: integer (nullable = true)
 |-- weight_lbs: string (nullable = true)



In [31]:
more_stats = get_string_length_stats(labeled_training_set)

In [32]:
more_stats.show()

+------------------+------------+------------+-----------------+-----------------+-----------------+-----------------+----------------------+------------------+----------+----------+-----------------+-------------------+-------------------+-------------------+------------------------+-----------------+---------------+---------------+--------------------+-----------------+----------------+----------------+---------------------+------------------+--------------+--------------+-------------------+-----------------+-----------------+-----------------+----------------------+------------------+-------------+-------------+-------------------+------------------+------------------+------------------+-----------------------+-------------------+-------------------+-------------------+------------------------+------------------+-----------------+-----------------+----------------------+------------------+------------------+------------------+-----------------------+-----------------+--------------

examples of dataframe api udf calls. Faster than sql, but will need to do more dataframe manipulation thats not as easy as spark.sql

examples of using udf in sql queries